# 01 — Load and clean LIMASAM container addresses

This notebook loads the raw container address list provided by LIMASAM (Limpieza de Málaga, S.A.M.) in March 2026, cleans the addresses for geocoding, and saves the result for the next step.

**Input:** `data/raw/limasam_containers_2026-03.xlsx` — one column (`Dirección:`), 483 free-text addresses.

**Output:** `data/interim/containers_cleaned.csv` — cleaned and tagged addresses, deduplicated, ready for geocoding.

## Cleaning strategy

After inspection, the raw addresses turned out to be highly heterogeneous: some include postcode and district, others only the street name; intersections are written in seven different ways (`esquina`, `esq.`, `con`, `&`, ...); abbreviations are inconsistent (`C/`, `C.`, `Av.`, `Avda.`, `Cmo`, `P.º` ...).

The following rules are applied:

1. **Expand street-type abbreviations**: `C/`, `C.` → `Calle`; `Av.`, `Avda.` → `Avenida`; `Ctra.` → `Carretera`; `Cmo`, `Cam.` → `Camino`; `P.º` → `Paseo`; `Pza` → `Plaza`; `Pje` → `Pasaje`.
2. **Normalize intersection words**: `esquina`, `esq.`, `con`, `&`, `entre ... y ...` are all rewritten to a single ` and ` so Nominatim sees a consistent format.
3. **Strip trailing descriptors**: phrases like `Frente parque infantil`, `Sobre acera`, `Cerca de glorieta`, parenthetical comments and `s/n` are removed — they only confuse the geocoder.
4. **Title-case** the result; small words (`de`, `la`, `del`) stay lowercase except when first.
5. **Append `, Málaga, Spain`** to every address.
6. **Tag each address** by structure: `house_number` (contains a digit), `intersection` (contains `and`-equivalent), or `street_only`.
7. **Deduplicate** twice: first by raw address (exact duplicates), then by cleaned address (case-insensitive duplicates that became identical after normalization).

In [1]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

print("Project root:    ", PROJECT_ROOT)
print("Raw data folder: ", DATA_RAW)
print("Interim folder:  ", DATA_INTERIM)

Project root:     c:\Users\user\Documents\Projects\malaga-textile-access
Raw data folder:  c:\Users\user\Documents\Projects\malaga-textile-access\data\raw
Interim folder:   c:\Users\user\Documents\Projects\malaga-textile-access\data\interim


In [2]:
# Load the Excel file
xlsx_path = DATA_RAW / "limasam_containers_2026-03.xlsx"
df = pd.read_excel(xlsx_path)

print("Shape (rows, columns):", df.shape)
print("Column names:", df.columns.tolist())
df.head(5)

Shape (rows, columns): (483, 1)
Column names: ['Dirección:']


,Dirección:
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad..."
3,"C. Federico García Lorca, 13-11, Carretera de ..."
4,"Calle Goya, 7, Carretera de Cádiz, 29002 Málaga"


In [3]:
col = "Dirección:"

# Missing and duplicates
print(f"Empty rows:       {df[col].isna().sum()}")
print(f"Exact duplicates: {df[col].duplicated().sum()}")

# Address length distribution
print(f"\nAddress length stats:")
print(df[col].str.len().describe())

# Show shortest and longest addresses
df_sorted = df.assign(length=df[col].str.len()).sort_values("length")

print(f"\n5 shortest addresses:")
for addr in df_sorted.head(5)[col].tolist():
    print(f"  {addr}")

print(f"\n5 longest addresses:")
for addr in df_sorted.tail(5)[col].tolist():
    print(f"  {addr}")

Empty rows:       0
Exact duplicates: 3

Address length stats:
count    483.000000
mean      31.739130
std       14.470254
min       10.000000
25%       21.000000
50%       28.000000
75%       38.500000
max       72.000000
Name: Dirección:, dtype: float64

5 shortest addresses:
  C/ ALMERIA
  FRANZ KAFKA
  C/ REAL, 15
  plaza babel
  C/ Praga, 7

5 longest addresses:
  rmoles, 14 Frente clínica "SINTESIS CENTER". Encima acera, al lado de k
  ntiago Ramón y Cajal, 67  Sobre acera. Altura parada de Taxis. Frente C
  tra. Sra. De Las Guías, 3 Esquina plaza Nazaret. Sobre acera, en Parada
  / Pastora Imperio, 11 Con C/ Sor Juana Inés de la Cruz. Cerca de gloriet
  n C/ Alcalde Joaquín Quiles. Frente parque infantil. Sobre acera de fren


In [4]:
import re

# How many addresses have no digit at all? Likely street-only or intersection.
has_number = df[col].str.contains(r"\d", regex=True)

print(f"Total addresses: {len(df)}")
print(f"  With at least one digit: {has_number.sum()}")
print(f"  Without any digit:        {(~has_number).sum()}")

no_number = df.loc[~has_number, col].tolist()
print(f"\nFirst 10 addresses without any digit (out of {len(no_number)}):")
for addr in no_number[:10]:
    print(f"  → {addr}")

Total addresses: 483
  With at least one digit: 386
  Without any digit:        97

First 10 addresses without any digit (out of 97):
  → C. Almogía MALAGA
  → Andarax con Camino de san Rafael
  → Aresnisca con Obsidiana - Malaga
  → C. Arlanzón MALAGA
  → Arroyo de los Angeles esquina Crisitino Martos
  → Arroyo de los Angeles esquina nuestra señora de los clarines
  → Avenida de Europa esquina puerto oncala
  → Av. de Molière MALAGA
  → Calle Bidasoa
  → C. Cancho Perez


## Cleaning functions

In [5]:
def expand_abbreviations(s: str) -> str:
    """Replace common Spanish street-type abbreviations with full words.
    
    Patterns absorb the trailing period so we don't leave orphan dots
    like 'Avenida.'. Order matters — longer patterns first.
    """
    rules = [
        # Avenida variants — match abbreviation with period or trailing space,
        # NEVER match 'Av' inside 'Avenida'
        (r"\bAvda\.",      "Avenida"),
        (r"\bAvda\b",      "Avenida"),
        (r"\bAv\.",        "Avenida"),
        (r"\bAv\s",        "Avenida "),
        # Other street types
        (r"\bCtra\.?",     "Carretera"),
        (r"\bCmno\.?",     "Camino"),
        (r"\bCmo\.?\s",    "Camino "),
        (r"\bCam\.\s",     "Camino "),
        (r"\bP\.º\s",      "Paseo "),
        (r"\bPza\.?",      "Plaza"),
        (r"\bPje\.?",      "Pasaje"),
        (r"\bC\.\s",       "Calle "),
        (r"\bC/\s?",       "Calle "),
        (r"\bCl\.?\b",     "Calle"),
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    # Collapse repeated whitespace
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_intersection_words(s: str) -> str:
    """Replace various intersection indicators with a single ' and '."""
    rules = [
        (r"\besquina\b", " and "),
        (r"\besq\.?\b",  " and "),
        (r"\s&\s",       " and "),
        (r"\bcon\b",     " and "),
        (r"\bentre\b",   ""),       # 'Entre X y Y' → ' X y Y'
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    # Clean repeated 'and'
    s = re.sub(r"\s+and\s+and\s+", " and ", s)
    return s


def strip_trailing_context(s: str) -> str:
    """Remove descriptive tails like 'Frente parque infantil', 'Sobre acera', 
    parenthetical comments, trailing 'Málaga', and 's/n'."""
    # Cut at common descriptor keywords
    cut_markers = [
        r"\bFrente\b",
        r"\bSobre\s+acera\b",
        r"\bCerca\s+de\b",
        r"\bLateral\b",
        r"\bJunto\b",
        r"\(",
    ]
    pattern = "|".join(cut_markers)
    s = re.split(pattern, s, maxsplit=1, flags=re.IGNORECASE)[0]
    
    # Remove trailing 'Málaga' (we add it back uniformly later)
    s = re.sub(r"[\s,;\-]*M[aá]laga\s*$", "", s, flags=re.IGNORECASE)
    
    # Remove 's/n' (sin número)
    s = re.sub(r",?\s*s/n\b", "", s, flags=re.IGNORECASE)
    
    # Tidy whitespace and punctuation
    s = re.sub(r"\s+", " ", s).strip(" ,-.;")
    return s


def smart_title_case(s: str) -> str:
    """Title-case the address, keeping small words lowercase except at start."""
    small = {"de", "del", "la", "las", "el", "los", "y", "and"}
    words = s.split()
    out = []
    for i, w in enumerate(words):
        if i > 0 and w.lower() in small:
            out.append(w.lower())
        else:
            out.append(w.capitalize())
    return " ".join(out)


def clean_address(raw: str) -> str:
    """Full cleaning pipeline for one raw address."""
    s = str(raw)
    s = strip_trailing_context(s)
    s = expand_abbreviations(s)
    s = normalize_intersection_words(s)
    s = smart_title_case(s)
    return f"{s}, Málaga, Spain"


def tag_address_type(raw: str) -> str:
    """Classify the raw address by structure: house_number / intersection / street_only."""
    s = raw.lower()
    has_digit = bool(re.search(r"\d", raw))
    has_intersection = bool(re.search(r"\b(esquina|esq\.?|con|&|entre)\b", s, flags=re.IGNORECASE))
    
    if has_digit:
        return "house_number"
    elif has_intersection:
        return "intersection"
    else:
        return "street_only"

In [6]:
# Apply cleaning + tagging
df_clean = df.copy()
df_clean["address_raw"] = df_clean[col]
df_clean["address_type"] = df_clean[col].apply(tag_address_type)
df_clean["address_clean"] = df_clean[col].apply(clean_address)

# Drop the original column
df_clean = df_clean.drop(columns=[col])

# Two-stage deduplication
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="address_raw").reset_index(drop=True)
after_raw = len(df_clean)
print(f"Stage 1: dropped {before - after_raw} exact duplicates "
      f"(matching on raw address)")

df_clean = df_clean.drop_duplicates(subset="address_clean").reset_index(drop=True)
after_clean = len(df_clean)
print(f"Stage 2: dropped {after_raw - after_clean} case-insensitive duplicates "
      f"(matching on cleaned address)")

print(f"\nFinal row count: {after_clean}")
print(f"\nAddress type distribution:")
print(df_clean["address_type"].value_counts())

print(f"\nSample (5 rows of each type):")
for t in ["house_number", "intersection", "street_only"]:
    print(f"\n--- {t} ---")
    print(df_clean[df_clean["address_type"] == t][["address_raw", "address_clean"]].head(3).to_string(index=False))

Stage 1: dropped 3 exact duplicates (matching on raw address)
Stage 2: dropped 2 case-insensitive duplicates (matching on cleaned address)

Final row count: 478

Address type distribution:
address_type
house_number    381
intersection     64
street_only      33
Name: count, dtype: int64

Sample (5 rows of each type):

--- house_number ---
                                                    address_raw                                                              address_clean
      Av. de la Paloma, 36-38, Carretera de Cádiz, 29003 Málaga      Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
         Av. de la Paloma, 43, Carretera de Cádiz, 29003 Málaga         Avenida de la Paloma, 43, Carretera de Cádiz, 29003, Málaga, Spain
C. Conde de Guadalhorce, 6-8, Cruz de Humilladero, 29006 Málaga Calle Conde de Guadalhorce, 6-8, Cruz de Humilladero, 29006, Málaga, Spain

--- intersection ---
                                   address_raw                                   

In [7]:
# Save cleaned addresses
out_path = DATA_INTERIM / "containers_cleaned.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8")

print(f"Saved {len(df_clean)} cleaned addresses to:")
print(f"  {out_path}")

# Verify by reading back
print(f"\nFirst 3 rows of saved file:")
pd.read_csv(out_path).head(3)

Saved 478 cleaned addresses to:
  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\containers_cleaned.csv

First 3 rows of saved file:


,address_raw,address_type,address_clean
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil..."
